In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from imports import *
from physics import *


# Physics

## Lagrangian <-> Hamiltonian Conversion

### 1D Standard Harmonic Oscillator

In [3]:
x, u, p = sp.symbols('x u p', real=True)
L_ho = 0.5 * p**2 - 0.5 * u**2

try:
    H_ho, (xi,) = LagrangianHamiltonianConverter.L_to_H(L_ho, (x,), u, (p,))
    print(f"  L = {L_ho}")
    print(f"  H (computed) = {H_ho}")
    print(f"  H (expected) = 1/2 xi^2 + 1/2 u^2")
    print(f"  Match: {sp.simplify(H_ho - (0.5 * xi**2 + 0.5 * u**2)) == 0}")
except Exception as e:
    print(f"  FAILED: {e}")

  L = 0.5*p**2 - 0.5*u**2
  H (computed) = 0.5*u**2 + 0.5*xi**2
  H (expected) = 1/2 xi^2 + 1/2 u^2
  Match: True


### 2D Free Particle

In [4]:
x, y, u, p_x, p_y = sp.symbols('x y u p_x p_y', real=True)
L_free = 0.5 * (p_x**2 + p_y**2)

try:
    H_free, (xi, eta) = LagrangianHamiltonianConverter.L_to_H(L_free, (x, y), u, (p_x, p_y))
    print(f"  L = {L_free}")
    print(f"  H (computed) = {H_free}")
    print(f"  H (expected) = 1/2 (xi^2 + eta^2)")
    print(f"  Match: {sp.simplify(H_free - 0.5 * (xi**2 + eta**2)) == 0}")
except Exception as e:
    print(f"  FAILED: {e}")

  L = 0.5*p_x**2 + 0.5*p_y**2
  H (computed) = 0.5*eta**2 + 0.5*xi**2
  H (expected) = 1/2 (xi^2 + eta^2)
  Match: True


### L -> H -> L Consistency (Harmonic Oscillator)

In [5]:
# Convert L -> H, then H -> L and check if we get back the original L (up to constant or sign)
try:
    L_orig = L_ho
    H_temp, (xi,) = LagrangianHamiltonianConverter.L_to_H(L_orig, (x,), u, (p,))
    L_back, (p_back,) = LagrangianHamiltonianConverter.H_to_L(H_temp, (x,), u, (xi,))
    print(f"  Original L = {L_orig}")
    print(f"  Reconstructed L = {L_back}")
    print(f"  Match: {sp.simplify(L_orig - L_back) == 0}")
except Exception as e:
    print(f"  FAILED: {e} (This might be expected due to asymmetry in H_to_L, see analysis)")

  Original L = 0.5*p**2 - 0.5*u**2
  Reconstructed L = 0.5*p**2 - 0.5*u**2
  Match: True


### L with Singular Hessian (p^4) - L->H failure

In [6]:
L_bad = p**4
try:
    H_bad, _ = LagrangianHamiltonianConverter.L_to_H(L_bad, (x,), u, (p,))
    print(f"  UNEXPECTED SUCCESS: H = {H_bad}")
except ValueError as e:
    print(f"  Expected failure occurred: {e}")
except Exception as e:
    print(f"  Unexpected error: {e}")

  UNEXPECTED SUCCESS: H = 3*2**(1/3)*xi**(4/3)/8


### Numeric Fenchel

In [7]:
L_fenchel = p**4 + p**2
try:
    H_repr, (xi,), H_num_func = LagrangianHamiltonianConverter.L_to_H(
        L_fenchel, (x,), u, (p,), method="fenchel_numeric"
    )
    print(f"  L = {L_fenchel}")
    print(f"  H (symbolic repr) = {H_repr}")
    print("  Sample H values:")
    for val in [-1.0, 0.0, 1.0]:
        h_val = H_num_func(val)
        print(f"    H(xi={val:.1f}) ≈ {h_val:.4f}")
except ImportError:
    print(f"  SKIPPED: SciPy not available for numeric Fenchel.")
except Exception as e:
    print(f"  FAILED: {e}")

  L = p**4 + p**2
  H (symbolic repr) = H_numeric(xi)
  Sample H values:
    H(xi=-1.0) ≈ 0.2148
    H(xi=0.0) ≈ -0.0000
    H(xi=1.0) ≈ 0.2148


## Hamiltonial to PDE Generation
### 1D Standard Kinetic + Potential

In [8]:
x, t, xi = sp.symbols("x t xi", real=True)
u = sp.Function("u")(t, x)
V = sp.Function("V")(x)
H_pde = 0.5 * xi**2 + V

try:
    pde_info = HamiltonianSymbolicConverter.hamiltonian_to_symbolic_pde(
        H_pde, (x,), t, u, mode="schrodinger"
    )
    print(f"  H = {H_pde}")
    print(f"  Schrödinger PDE: {pde_info['pde']}")
    print(f"  Formal string: {pde_info['formal_string']}")
except Exception as e:
    print(f"  FAILED: {e}")

  H = 0.5*xi**2 + V(x)
  Schrödinger PDE: Eq(I*Derivative(u(t, x), t), psiOp(0.5*xi**2 + V(x), u(t, x)))
  Formal string: i ∂_t u = ψOp(H, u)   (H = H(x; xi))


### 2D Kinetic + Potential

In [9]:
x, y, t = sp.symbols("x y t", real=True)
u2 = sp.Function("u")(t, x, y)
xi, eta = sp.symbols("xi eta", real=True)
V2 = sp.Function("V")(x, y)
H2D_pde = 0.5 * (xi**2 + eta**2) + V2

try:
    pde_info_2d = HamiltonianSymbolicConverter.hamiltonian_to_symbolic_pde(
        H2D_pde, (x, y), t, u2, mode="wave"
    )
    print(f"  H = {H2D_pde}")
    print(f"  Wave PDE: {pde_info_2d['pde']}")
    print(f"  Formal string: {pde_info_2d['formal_string']}")
except Exception as e:
    print(f"  FAILED: {e}")

  H = 0.5*eta**2 + 0.5*xi**2 + V(x, y)
  Wave PDE: Eq(Derivative(u(t, x, y), (t, 2)), -psiOp(0.5*eta**2 + 0.5*xi**2 + V(x, y), u(t, x, y)))
  Formal string: ∂_{tt} u + ψOp(H, u) = 0   (H = H(x, y; xi, eta))
